# 🥈 Silver Pipeline — Bronze → `jobs_clean_silver`

**Purpose**: 
- Deduplicate jobs (same job from multiple portals → 1 silver row)
- Parse experience string → `experience_min` / `experience_max` INT
- AI enrich: `roles_summary`, `ai_summary`, `validation_score`, `tech_stack`
- Mark stale jobs (not seen 7+ days) as `active = FALSE`

**Dedup key**: `job_hash = MD5(lower(company_name + job_title))`

In [0]:
import os, re, json, uuid, requests, time
from datetime import date, timedelta, datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, lit, current_date, current_timestamp, udf, when, split,
    regexp_extract, coalesce, to_date, trim
)
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

CATALOG        = "jobs_automation_db"
BRONZE_TABLE   = f"{CATALOG}.default.jobs_harvested_bronze"
SILVER_TABLE   = f"{CATALOG}.default.jobs_clean_silver"

# NVIDIA NIM API
NVIDIA_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nim")
NVIDIA_MODEL   = "meta/llama-3.1-8b-instruct"
NVIDIA_URL     = "https://integrate.api.nvidia.com/v1/chat/completions"

# Only AI-score jobs with description >= this length (words)
MIN_DESC_WORDS = 50

TODAY = date.today()
STALE_AFTER_DAYS = 7

print(f"✅ Config ready. Silver table: {SILVER_TABLE}")

In [0]:
# Load bronze jobs not yet in silver
try:
    silver_hashes = spark.sql(f"SELECT job_hash FROM {SILVER_TABLE}").rdd.flatMap(lambda x: x).collect()
    silver_hashes_set = set(silver_hashes)
    print(f"📊 Existing silver jobs: {len(silver_hashes_set)}")
except Exception:
    silver_hashes_set = set()
    print("ℹ️  Silver table empty — first run")

bronze_df = spark.sql(f"""
    SELECT 
        job_hash, job_title, company_name, location, remote_type,
        salary_range, experience_years, tech_stack,
        job_description, roles_responsibilities, requirements_section,
        roles_summary, ai_summary, apply_link, easy_apply_link,
        company_career_url, company_website, hr_email,
        portal, posted_date, fetch_date, visa_sponsorship,
        validation_score
    FROM {BRONZE_TABLE}
    WHERE job_hash IS NOT NULL
      AND job_hash != ''
      AND job_title IS NOT NULL
""")

# Deduplicate bronze itself (keep first occurrence per job_hash)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("job_hash").orderBy(desc("fetch_date"))
bronze_dedup = bronze_df.withColumn("rn", row_number().over(window)) \
                        .filter(col("rn") == 1).drop("rn")

# Filter to only new jobs
new_jobs_df = bronze_dedup.filter(~col("job_hash").isin(silver_hashes_set))
new_count = new_jobs_df.count()
print(f"🆕 New jobs to process: {new_count}")

In [0]:
def parse_exp_range(text: str):
    """Extract (min, max) integer years from string like '5-8 years' or '5+ years'."""
    if not text:
        return (0, 99)
    text = text.lower().strip()
    
    # Match: 5-8 years / 5 to 8 years
    m = re.search(r'(\d+)\s*(?:-|to)\s*(\d+)\s*years?', text)
    if m:
        return (int(m.group(1)), int(m.group(2)))
    
    # Match: 5+ years
    m = re.search(r'(\d+)\+\s*years?', text)
    if m:
        return (int(m.group(1)), 99)
    
    # Match: 5 years
    m = re.search(r'(\d+)\s*years?', text)
    if m:
        return (int(m.group(1)), int(m.group(1)) + 2)
    
    return (0, 99)

parse_exp_min_udf = udf(lambda t: parse_exp_range(t)[0], IntegerType())
parse_exp_max_udf = udf(lambda t: parse_exp_range(t)[1], IntegerType())

new_jobs_df = new_jobs_df \
    .withColumn("experience_min", parse_exp_min_udf(col("experience_years"))) \
    .withColumn("experience_max", parse_exp_max_udf(col("experience_years")))

print("✅ Experience parsed into experience_min / experience_max")

In [0]:
def ai_enrich_job(title, company, location, desc, existing_tech, existing_summary):
    """
    Call NVIDIA NIM to:
    - Score job relevance (0-100)
    - Generate roles_summary (3-5 bullet points)
    - Extract/improve tech_stack
    - Write ai_summary (2 sentences)
    - Detect visa sponsorship, remote type
    Returns dict with enriched fields.
    """
    desc_trimmed = (desc or "")[:3000]
    
    # Skip if desc too short — use rule-based score
    if len(desc_trimmed.split()) < MIN_DESC_WORDS:
        return {
            "validation_score": 50,
            "validation_status": "Partial",
            "ai_summary": f"Senior {title} role at {company}. Description not available.",
            "roles_summary": "",
            "tech_stack": existing_tech or "",
            "remote_type": "Not specified",
            "visa_sponsorship": None,
        }

    prompt = f"""Analyze this US IT job posting. Respond ONLY with valid JSON, no markdown.

Title: {title}
Company: {company}
Location: {location}
Description:
{desc_trimmed}

Respond with this exact JSON:
{{
  "score": <0-100 integer, how relevant for US IT job seeker>,
  "is_real_job": <true/false>,
  "ai_summary": "<2 sentence summary of the role>",
  "roles_summary": "<3-5 key responsibilities as bullet points starting with •>",
  "tech_stack": "<top 10 technical skills comma-separated>",
  "experience_years": "<e.g. '5+ years' or 'Not specified'>",
  "remote_type": "<Remote|Hybrid|Onsite|Not specified>",
  "visa_sponsorship": <true/false/null>
}}"""

    for attempt in range(3):
        try:
            resp = requests.post(
                NVIDIA_URL,
                headers={"Authorization": f"Bearer {NVIDIA_API_KEY}",
                         "Content-Type": "application/json"},
                json={"model": NVIDIA_MODEL,
                      "messages": [{"role": "user", "content": prompt}],
                      "temperature": 0.1,
                      "max_tokens": 600},
                timeout=25,
            )
            if resp.status_code == 200:
                content = resp.json()["choices"][0]["message"]["content"].strip()
                m = re.search(r'\{.*\}', content, re.DOTALL)
                if m:
                    data = json.loads(m.group())
                    score = int(data.get("score", 60))
                    # If not a real job posting, score it 0
                    if not data.get("is_real_job", True):
                        score = 0
                    return {
                        "validation_score": score,
                        "validation_status": "Valid" if score >= 70 else "Partial" if score >= 40 else "Junk",
                        "ai_summary": data.get("ai_summary", ""),
                        "roles_summary": data.get("roles_summary", ""),
                        "tech_stack": data.get("tech_stack") or existing_tech or "",
                        "remote_type": data.get("remote_type", "Not specified"),
                        "visa_sponsorship": data.get("visa_sponsorship"),
                    }
            elif resp.status_code == 429:
                time.sleep(10 * (attempt + 1))
        except json.JSONDecodeError:
            pass
        except Exception as e:
            print(f"AI error (attempt {attempt+1}): {e}")
            time.sleep(5)

    return {
        "validation_score": 55,
        "validation_status": "Partial",
        "ai_summary": "AI enrichment unavailable.",
        "roles_summary": "",
        "tech_stack": existing_tech or "",
        "remote_type": "Not specified",
        "visa_sponsorship": None,
    }

# Process jobs — collect to driver for AI calls
new_jobs_list = new_jobs_df.collect()
print(f"🤖 AI enriching {len(new_jobs_list)} new jobs with NVIDIA NIM...")

enriched_records = []
for i, row in enumerate(new_jobs_list):
    enrichment = ai_enrich_job(
        row.job_title, row.company_name, row.location,
        row.job_description, row.tech_stack, row.ai_summary
    )
    record = row.asDict()
    record.update(enrichment)
    # Set silver-specific fields
    record["first_seen_date"] = TODAY
    record["last_seen_date"]  = TODAY
    record["active"]          = True
    enriched_records.append(record)

    if (i + 1) % 20 == 0:
        print(f"   Progress: {i+1}/{len(new_jobs_list)} enriched...")

print(f"✅ AI enrichment complete for {len(enriched_records)} jobs")

In [0]:
from pyspark.sql.types import BooleanType, IntegerType, DateType, StringType, StructType, StructField

SILVER_SCHEMA = StructType([
    StructField("job_hash",              StringType(),  True),
    StructField("job_title",             StringType(),  True),
    StructField("company_name",          StringType(),  True),
    StructField("location",              StringType(),  True),
    StructField("remote_type",           StringType(),  True),
    StructField("salary_range",          StringType(),  True),
    StructField("experience_min",        IntegerType(), True),
    StructField("experience_max",        IntegerType(), True),
    StructField("tech_stack",            StringType(),  True),
    StructField("job_description",       StringType(),  True),
    StructField("roles_responsibilities",StringType(),  True),
    StructField("requirements_section",  StringType(),  True),
    StructField("roles_summary",         StringType(),  True),
    StructField("ai_summary",            StringType(),  True),
    StructField("apply_link",            StringType(),  True),
    StructField("easy_apply_link",       StringType(),  True),
    StructField("company_career_url",    StringType(),  True),
    StructField("company_website",       StringType(),  True),
    StructField("hr_email",              StringType(),  True),
    StructField("portal",                StringType(),  True),
    StructField("posted_date",           StringType(),  True),
    StructField("fetch_date",            DateType(),    True),
    StructField("visa_sponsorship",      BooleanType(), True),
    StructField("validation_score",      IntegerType(), True),
    StructField("validation_status",     StringType(),  True),
    StructField("first_seen_date",       DateType(),    True),
    StructField("last_seen_date",        DateType(),    True),
    StructField("active",                BooleanType(), True),
])

if enriched_records:
    silver_new_df = spark.createDataFrame(enriched_records, schema=SILVER_SCHEMA)

    # Filter out junk jobs
    silver_new_df = silver_new_df.filter(col("validation_status") != "Junk")
    print(f"📊 Jobs after quality filter: {silver_new_df.count()}")

    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

    (
        silver_delta.alias("target")
        .merge(
            silver_new_df.alias("source"),
            "target.job_hash = source.job_hash"
        )
        # New job → insert all
        .whenNotMatchedInsertAll()
        # Existing job seen again → update last_seen_date and active status
        .whenMatchedUpdate(set={
            "target.last_seen_date": "source.last_seen_date",
            "target.active": "true",
            # Update hr_email if we have a better one
            "target.hr_email": "CASE WHEN source.hr_email != '' THEN source.hr_email ELSE target.hr_email END",
        })
        .execute()
    )
    print(f"✅ Silver MERGE complete!")
else:
    print("ℹ️  No new jobs to merge into silver.")

In [0]:
# Mark jobs not seen in 7+ days as active=False
stale_cutoff = (date.today() - timedelta(days=STALE_AFTER_DAYS)).isoformat()
spark.sql(f"""
    UPDATE {SILVER_TABLE}
    SET active = false
    WHERE last_seen_date < '{stale_cutoff}'
    AND active = true
""")
print(f"✅ Stale jobs marked inactive (not seen since {stale_cutoff})")

In [0]:
%sql
SELECT
    active,
    remote_type,
    COUNT(*) AS jobs,
    AVG(validation_score) AS avg_score,
    COUNT(DISTINCT company_name) AS companies
FROM jobs_automation_db.default.jobs_clean_silver
GROUP BY active, remote_type
ORDER BY jobs DESC